In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from datetime import datetime, timedeltaimport sqlite3



print(" Starting Flipkart Data Extraction & Pipeline...")

# ==========================================
# STEP 1: Web Scraping Base Data from Flipkart
# ==========================================
url = "https://www.flipkart.com/search?q=laptops&page=1"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US, en;q=0.5"
}

response = requests.get(url, headers=headers)
product_names, prices, ratings = [], [], []

if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')
    containers = soup.find_all('div', {'class': '_75nlfW'})
    
    for container in containers:
        name_elem = container.find('div', {'class': 'KzDlHZ'})
        price_elem = container.find('div', {'class': 'Nx9bqj _4b5DiR'})
        rating_elem = container.find('div', {'class': 'XQDdHH'})
        
        if name_elem and price_elem:
            product_names.append(name_elem.text)
            p_clean = price_elem.text.replace('₹', '').replace(',', '').strip()
            prices.append(int(p_clean) if p_clean.isdigit() else 50000)
            ratings.append(float(rating_elem.text) if rating_elem else 4.0)

# 🛠️ Safety Check: Agar scraping block ho gayi ya 0 products aaye, toh standard Flipkart products use karein
if len(product_names) == 0:
    print(" Web scraping blocked by Flipkart. Using standard Flipkart laptop catalog for pipeline...")
    product_names = [
        "HP Pavilion Core i5 13th Gen", "ASUS Vivobook 15 Intel Core i3", 
        "Lenovo IdeaPad Slim 3", "Apple MacBook Air M2", 
        "Dell Inspiron 3520", "Acer Aspire 5 Gaming", "MSI Thin GF63"
    ]
    prices = [64990, 37990, 41990, 99990, 45990, 52990, 57990]
    ratings = [4.3, 4.0, 4.2, 4.8, 4.1, 4.3, 4.2]

print(f" Using {len(product_names)} base products for dataset generation.")

# ==========================================
# STEP 2: Scaling & Simulating 1000+ Rows for Company Analytics
# ==========================================
np.random.seed(42)
n_rows = 1500  # 1500+ rows ka bada dataset

sampled_indices = np.random.randint(0, len(product_names), n_rows)
final_products = [product_names[i] for i in sampled_indices]
final_prices = [prices[i] for i in sampled_indices]

# October 2026 Dates Generation
start_date = datetime(2026, 10, 1)
order_dates = [start_date + timedelta(days=int(np.random.randint(0, 31)), 
                                      hours=int(np.random.randint(0, 24)), 
                                      minutes=int(np.random.randint(0, 60))) for _ in range(n_rows)]

# City & Country Performance
cities = ['Delhi', 'Mumbai', 'Bangalore', 'Hyderabad', 'Chennai', 'Kolkata', 'Pune', 'Bareilly', 'Ahmedabad', 'Jaipur']
city_weights = [0.18, 0.18, 0.12, 0.10, 0.08, 0.08, 0.08, 0.06, 0.06, 0.06]
order_cities = np.random.choice(cities, n_rows, p=city_weights)

# Order Status & Cancellation Reasons Analysis
statuses = ['Delivered', 'Cancelled', 'Returned']
order_statuses = np.random.choice(statuses, n_rows, p=[0.76, 0.14, 0.10])

cancellation_reasons = []
for status in order_statuses:
    if status == 'Cancelled':
        reason = np.random.choice(['Out of Stock', 'Customer Changed Mind', 'Payment Failure', 'Found Cheaper Elsewhere'], p=[0.3, 0.4, 0.15, 0.15])
    elif status == 'Returned':
        reason = np.random.choice(['Defective Product', 'Wrong Item Delivered', 'Size/Quality Issue'], p=[0.5, 0.3, 0.2])
    else:
        reason = 'None (Successful Delivery)'
    cancellation_reasons.append(reason)

# Customer IDs & Delivery Timelines
customer_pool = [f"CUST_{np.random.randint(5000, 5300)}" for _ in range(300)]
customer_ids = np.random.choice(customer_pool, n_rows)
delivery_days = np.random.randint(2, 8, n_rows)

# ==========================================
# STEP 3: Creating Master DataFrame & Exporting
# ==========================================
df_master = pd.DataFrame({
    'Order_ID': [f"ORD_{10000 + i}" for i in range(n_rows)],
    'Customer_ID': customer_ids,
    'Order_Date': order_dates,
    'Product_Name': final_products,
    'Unit_Price': final_prices,
    'Quantity': np.random.choice([1, 2, 3], n_rows, p=[0.75, 0.20, 0.05]),
    'City': order_cities,
    'Country': 'India',
    'Order_Status': order_statuses,
    'Cancellation_Reason': cancellation_reasons,
    'Delivery_Days': delivery_days
})

df_master['Total_Revenue'] = df_master['Unit_Price'] * df_master['Quantity']

file_name = 'Flipkart_October_2026_Performance_Report.csv'
df_master.to_csv(file_name, index=False)

print(f"\n Pipeline Executed Successfully!")
print(f" Total Records Generated: {len(df_master)}")
print(f" Saved file as: {file_name}")
print("\n--- Preview of Dataset ---")
print(df_master[['Order_ID', 'Order_Date', 'City', 'Order_Status', 'Total_Revenue']].head(5))

 Starting Flipkart Data Extraction & Pipeline...
 Web scraping blocked by Flipkart. Using standard Flipkart laptop catalog for pipeline...
 Using 7 base products for dataset generation.

 Pipeline Executed Successfully!
 Total Records Generated: 1500
 Saved file as: Flipkart_October_2026_Performance_Report.csv

--- Preview of Dataset ---
    Order_ID          Order_Date       City Order_Status  Total_Revenue
0  ORD_10000 2026-10-30 16:44:00  Bangalore    Delivered         115980
1  ORD_10001 2026-10-29 15:39:00  Bangalore    Delivered          99990
2  ORD_10002 2026-10-19 17:00:00    Chennai    Delivered          45990
3  ORD_10003 2026-10-14 18:46:00    Chennai    Cancelled          57990
4  ORD_10004 2026-10-18 01:53:00  Bangalore    Cancelled          41990


In [3]:
print(df_master.columns.tolist())

['Order_ID', 'Customer_ID', 'Order_Date', 'Product_Name', 'Unit_Price', 'Quantity', 'City', 'Country', 'Order_Status', 'Cancellation_Reason', 'Delivery_Days', 'Total_Revenue']


In [5]:
# 1. Load the generated CSV file
file_name = 'Flipkart_October_2026_Performance_Report.csv'
df = pd.read_csv(file_name)

print("--- 1. Initial Dataset Overview ---")
print(f"Total Rows (Transactions): {df.shape[0]}")
print(f"Total Columns (Attributes): {df.shape[1]}")

# 2. Check for Missing / Null Values
print("\n--- 2. Missing Values Check ---")
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0] if null_counts.sum() > 0 else "No missing values found! Data is clean.")

# 3. Data Type Inspection & Cleaning
print("\n--- 3. Fixing Data Types ---")
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['Unit_Price'] = pd.to_numeric(df['Unit_Price'], errors='coerce')
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Total_Revenue'] = pd.to_numeric(df['Total_Revenue'], errors='coerce')

# 4. Feature Engineering (Corporate Level Additions)
df['Month'] = df['Order_Date'].dt.strftime('%Y-%m')
df['Day_Name'] = df['Order_Date'].dt.day_name()
df['Revenue_Category'] = pd.qcut(df['Total_Revenue'], q=3, labels=['Low', 'Medium', 'High'])

print("\n--- 4. Data Cleaning & Validation Completed Successfully! ---")
display(df.head(3))

--- 1. Initial Dataset Overview ---
Total Rows (Transactions): 1500
Total Columns (Attributes): 12

--- 2. Missing Values Check ---
No missing values found! Data is clean.

--- 3. Fixing Data Types ---

--- 4. Data Cleaning & Validation Completed Successfully! ---


,Order_ID,Customer_ID,Order_Date,Product_Name,Unit_Price,Quantity,City,Country,Order_Status,Cancellation_Reason,Delivery_Days,Total_Revenue,Month,Day_Name,Revenue_Category
0,ORD_10000,CUST_5153,2026-10-30 16:44:00,MSI Thin GF63,57990,2,Bangalore,India,Delivered,None (Successful Delivery),5,115980,2026-10,Friday,High
1,ORD_10001,CUST_5244,2026-10-29 15:39:00,Apple MacBook Air M2,99990,1,Bangalore,India,Delivered,None (Successful Delivery),5,99990,2026-10,Thursday,High
2,ORD_10002,CUST_5076,2026-10-19 17:00:00,Dell Inspiron 3520,45990,1,Chennai,India,Delivered,None (Successful Delivery),3,45990,2026-10,Monday,Low


In [7]:
import sqlite3
# 1. Establish SQLite Connection & Database Creation
conn = sqlite3.connect('flipkart_corporate_analytics.db')
cursor = conn.cursor()

# 2. Load Cleaned DataFrame into SQLite Table
df.to_sql('corporate_orders', conn, if_exists='replace', index=False)
print(" Success: Data successfully loaded into SQLite database table 'corporate_orders'\n")

# ==========================================
# PHASE 3: CORPORATE BUSINESS QUERIES & INSIGHTS
# ==========================================

print("="*60)
print(" EXECUTIVE SUMMARY: Top Performing Cities by Revenue")
print("="*60)
query_city_perf = """
    SELECT City, 
           COUNT(Order_ID) AS Total_Orders, 
           SUM(Total_Revenue) AS Gross_Revenue,
           ROUND(AVG(Delivery_Days), 1) AS Avg_Delivery_Time
    FROM corporate_orders
    GROUP BY City
    ORDER BY Gross_Revenue DESC
    LIMIT 5;
"""
df_city = pd.read_sql(query_city_perf, conn)
display(df_city)


print("\n" + "="*60)
print(" SUPPLY CHAIN LOSS ANALYSIS: Cancellation & Return Breakdown")
print("="*60)
query_cancellation = """
    SELECT Order_Status, 
           Cancellation_Reason, 
           COUNT(Order_ID) AS Incident_Count,
           ROUND(SUM(Total_Revenue), 2) AS Revenue_At_Risk
    FROM corporate_orders
    WHERE Order_Status != 'Delivered'
    GROUP BY Order_Status, Cancellation_Reason
    ORDER BY Incident_Count DESC;
"""
df_cancel = pd.read_sql(query_cancellation, conn)
display(df_cancel)


print("\n" + "="*60)
print("CUSTOMER RETENTION: Repeat Buyers Analysis")
print("="*60)
query_retention = """
    SELECT Customer_ID, 
           COUNT(Order_ID) AS Total_Orders_Placed,
           SUM(Total_Revenue) AS Customer_Lifetime_Value
    FROM corporate_orders
    GROUP BY Customer_ID
    HAVING Total_Orders_Placed > 1
    ORDER BY Total_Orders_Placed DESC
    LIMIT 5;
"""
df_retention = pd.read_sql(query_retention, conn)
display(df_retention)

# Close database connection
conn.close()

 Success: Data successfully loaded into SQLite database table 'corporate_orders'

 EXECUTIVE SUMMARY: Top Performing Cities by Revenue


,City,Total_Orders,Gross_Revenue,Avg_Delivery_Time
0,Mumbai,293,21648190,4.6
1,Delhi,275,21518210,4.5
2,Bangalore,167,12071940,4.5
3,Hyderabad,137,9566250,4.5
4,Chennai,129,9226400,4.9



 SUPPLY CHAIN LOSS ANALYSIS: Cancellation & Return Breakdown


,Order_Status,Cancellation_Reason,Incident_Count,Revenue_At_Risk
0,Cancelled,Customer Changed Mind,75,5400000.0
1,Returned,Defective Product,69,4815150.0
2,Cancelled,Out of Stock,60,4715180.0
3,Returned,Wrong Item Delivered,52,3625350.0
4,Cancelled,Found Cheaper Elsewhere,32,2219600.0
5,Cancelled,Payment Failure,31,1985630.0
6,Returned,Size/Quality Issue,19,1455760.0



CUSTOMER RETENTION: Repeat Buyers Analysis


,Customer_ID,Total_Orders_Placed,Customer_Lifetime_Value
0,CUST_5270,32,2607580
1,CUST_5019,27,1927670
2,CUST_5022,23,1791680
3,CUST_5155,19,1218800
4,CUST_5007,19,1085800


In [8]:
df_master.to_csv('Flipkart_Clean_Performance_Report.csv', index=False)

print(" CSV file successfully saved !")

 CSV file successfully saved in your Jupyter Notebook directory!
